# Complicated Momentum With Strict Sharpe Threshold

This notebook revisits the richer multi-horizon momentum idea, but adds a strict training-Sharpe threshold for asset inclusion.

Workflow:

1. Build risk-adjusted cross-sectional momentum z-scores for lookbacks `[5, 10, 20, 30, 40, 50, 60, 70]`.
2. For each asset, search for horizon weights `a` using only the first 400 days.
3. Apply a strict training-Sharpe threshold to decide which assets are allowed into the portfolio.
4. Report the last 100 days only as an out-of-sample sanity check.

`teamName.py` is not changed by this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

prices = pd.read_csv("prices.txt", sep=r"\s+")
assets = list(prices.columns)
returns = prices.pct_change()

LOOKBACKS = [5, 10, 20, 30, 40, 50, 60, 70]
TRAIN_END = 400
TEST_START = 400
RANDOM_SEED = 20260709
N_RANDOM_CANDIDATES = 2400

DEFAULT_POSITION_LIMIT = 10_000
SPECIAL_POSITION_LIMIT = 100_000
DEFAULT_COMMISSION = 0.0001
SPECIAL_COMMISSION = 0.00002

position_limits = pd.Series(DEFAULT_POSITION_LIMIT, index=assets, dtype=float)
position_limits.iloc[0] = SPECIAL_POSITION_LIMIT

commission_rates = pd.Series(DEFAULT_COMMISSION, index=assets, dtype=float)
commission_rates.iloc[0] = SPECIAL_COMMISSION

prices.tail()

In [ ]:
def score_function(mean_pl, std_pl):
    if mean_pl <= 0 or std_pl < 1e-10:
        return mean_pl
    sharpe = np.sqrt(250) * mean_pl / std_pl
    return mean_pl * (sharpe**2 / (sharpe**2 + 1))


def performance_summary(daily_pl):
    daily_pl = pd.Series(daily_pl).replace([np.inf, -np.inf], np.nan).dropna()
    mean_pl = daily_pl.mean()
    std_pl = daily_pl.std(ddof=0)
    sharpe = 0.0 if std_pl <= 0 else np.sqrt(250) * mean_pl / std_pl
    return pd.Series({
        "mean_pl": mean_pl,
        "std_pl": std_pl,
        "annualised_sharpe": sharpe,
        "score": score_function(mean_pl, std_pl),
        "total_pl": daily_pl.sum(),
    })


def annualised_sharpe(series):
    series = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(series) < 2:
        return -np.inf
    volatility = series.std(ddof=0)
    if not np.isfinite(volatility) or volatility <= 0:
        return -np.inf
    return np.sqrt(250) * series.mean() / volatility


def make_z_scores(returns):
    past_returns = returns.shift(1)
    z_scores = {}

    for lookback in LOOKBACKS:
        momentum = past_returns.rolling(lookback).sum()
        volatility = past_returns.rolling(lookback).std()
        vol_normalised = momentum / volatility.replace(0, np.nan)
        cross_mean = vol_normalised.mean(axis=1)
        cross_std = vol_normalised.std(axis=1).replace(0, np.nan)
        z_scores[lookback] = (
            vol_normalised
            .sub(cross_mean, axis=0)
            .div(cross_std, axis=0)
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

    return z_scores


def candidate_weights():
    rng = np.random.default_rng(RANDOM_SEED)
    candidates = [np.ones(len(LOOKBACKS)) / len(LOOKBACKS)]

    for index in range(len(LOOKBACKS)):
        one_hot = np.zeros(len(LOOKBACKS))
        one_hot[index] = 1.0
        candidates.append(one_hot)

    for alpha in [0.5, 1.0, 2.0, 4.0, 8.0]:
        draws = rng.dirichlet(np.full(len(LOOKBACKS), alpha), size=N_RANDOM_CANDIDATES // 5)
        candidates.extend(draws)

    return np.array(candidates)


## Optimise Horizon Weights Per Asset

For each asset, we choose the horizon weights with the best regularised training Sharpe. The regularisation mildly penalises concentrated / one-hot `a` vectors.

In [ ]:
z_scores = make_z_scores(returns)
candidates = candidate_weights()
candidate_concentration = np.sum(candidates**2, axis=1)
candidate_max_weight = np.max(candidates, axis=1)

CONCENTRATION_PENALTY = 0.02
MAX_WEIGHT_PENALTY = 0.02

asset_rows = []
asset_signals = {}

for asset in assets:
    asset_z = np.column_stack([z_scores[lookback][asset].to_numpy() for lookback in LOOKBACKS])
    asset_returns = returns[asset].to_numpy()
    strategy_returns = (asset_z @ candidates.T) * asset_returns[:, None]

    train_returns = strategy_returns[:TRAIN_END]
    train_means = np.nanmean(train_returns, axis=0)
    train_stds = np.nanstd(train_returns, axis=0, ddof=0)
    train_sharpes = np.divide(
        np.sqrt(250) * train_means,
        train_stds,
        out=np.full_like(train_means, -np.inf),
        where=train_stds > 0,
    )

    objectives = (
        train_sharpes
        - CONCENTRATION_PENALTY * candidate_concentration
        - MAX_WEIGHT_PENALTY * candidate_max_weight
    )
    best_index = int(np.nanargmax(objectives))
    best_weights = candidates[best_index]

    asset_signal = pd.Series(asset_z @ best_weights, index=prices.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    asset_signals[asset] = asset_signal

    test_returns = strategy_returns[TEST_START:, best_index]
    row = {
        "asset": asset,
        "train_sharpe": train_sharpes[best_index],
        "test_sharpe": annualised_sharpe(test_returns),
        "concentration": candidate_concentration[best_index],
        "max_weight": candidate_max_weight[best_index],
    }
    row.update({f"a{lookback}": best_weights[i] for i, lookback in enumerate(LOOKBACKS)})
    asset_rows.append(row)

asset_table = pd.DataFrame(asset_rows).sort_values("train_sharpe", ascending=False).reset_index(drop=True)
signal = pd.DataFrame(asset_signals)

display(asset_table)

## Portfolio Backtest Helpers

In [ ]:
def signal_to_dollar_targets(signal_frame, selected_assets):
    selected_signal = signal_frame[selected_assets].copy()
    gross_signal = selected_signal.abs().sum(axis=1).replace(0, np.nan)
    weights = selected_signal.div(gross_signal, axis=0).fillna(0.0)

    targets = pd.DataFrame(0.0, index=signal_frame.index, columns=signal_frame.columns)
    targets[selected_assets] = weights * position_limits[selected_assets]
    return targets.clip(lower=-position_limits, upper=position_limits, axis=1)


def backtest_dollar_targets(dollar_targets, start_day, end_day):
    share_targets = (dollar_targets / prices).replace([np.inf, -np.inf], np.nan).fillna(0).astype(int)

    cash = 0.0
    current_position = pd.Series(0.0, index=assets)
    portfolio_value = 0.0
    daily_pl = []
    daily_values = []

    for t in range(start_day, end_day + 1):
        current_prices = prices.iloc[t]
        if t < end_day:
            new_position = share_targets.iloc[t].astype(float)
        else:
            new_position = current_position.copy()

        trade = new_position - current_position
        traded_dollars = current_prices * trade.abs()
        commission = (traded_dollars * commission_rates).sum()

        cash -= current_prices.dot(trade) + commission
        current_position = new_position

        new_value = cash + current_position.dot(current_prices)
        if t > start_day:
            daily_pl.append(new_value - portfolio_value)
            daily_values.append(new_value)
        portfolio_value = new_value

    index = prices.index[start_day + 1:end_day + 1]
    return {
        "daily_pl": pd.Series(daily_pl, index=index, name="daily_pl"),
        "daily_values": pd.Series(daily_values, index=index, name="value"),
        "final_value": portfolio_value,
    }


## Strict Training-Sharpe Threshold Sweep

In [ ]:
threshold_rows = []
thresholds = [0.0, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00, 2.50, 3.00]

for threshold in thresholds:
    selected_assets = asset_table.loc[asset_table["train_sharpe"] > threshold, "asset"].tolist()
    if not selected_assets:
        continue

    targets = signal_to_dollar_targets(signal, selected_assets)
    train_result = backtest_dollar_targets(targets, start_day=max(LOOKBACKS) + 2, end_day=TRAIN_END - 1)
    test_result = backtest_dollar_targets(targets, start_day=TEST_START, end_day=len(prices) - 1)

    train_summary = performance_summary(train_result["daily_pl"])
    test_summary = performance_summary(test_result["daily_pl"])

    threshold_rows.append({
        "threshold": threshold,
        "n_assets": len(selected_assets),
        "assets": selected_assets,
        "train_sharpe": train_summary["annualised_sharpe"],
        "train_score": train_summary["score"],
        "test_sharpe": test_summary["annualised_sharpe"],
        "test_score": test_summary["score"],
        "test_total_pl": test_summary["total_pl"],
    })

threshold_table = pd.DataFrame(threshold_rows).sort_values("threshold").reset_index(drop=True)
display(threshold_table.drop(columns=["assets"]))

best_by_train_score = threshold_table.sort_values("train_score", ascending=False).iloc[0]
strict_assets = best_by_train_score["assets"]

print("Best threshold by training score:", best_by_train_score["threshold"])
print(f"Selected {len(strict_assets)} assets:")
print(strict_assets)

In [ ]:
strict_targets = signal_to_dollar_targets(signal, strict_assets)
strict_train = backtest_dollar_targets(strict_targets, start_day=max(LOOKBACKS) + 2, end_day=TRAIN_END - 1)
strict_test = backtest_dollar_targets(strict_targets, start_day=TEST_START, end_day=len(prices) - 1)

summary = pd.DataFrame({
    "strict_train": performance_summary(strict_train["daily_pl"]),
    "strict_test": performance_summary(strict_test["daily_pl"]),
})
display(summary)

fig, ax = plt.subplots(figsize=(10, 4))
strict_train["daily_pl"].cumsum().plot(ax=ax, label="strict train cumulative PL")
strict_test["daily_pl"].cumsum().plot(ax=ax, label="strict test cumulative PL")
ax.axhline(0, color="black", linewidth=1, alpha=0.4)
ax.set_title("Strict complicated momentum cumulative PL")
ax.set_ylabel("Cumulative PL")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()